In [ ]:
from pathlib import Path
import os
import cdsapi
import calendar
import time
from datetime import datetime

This script downloads the requested ERA5 data from the Climate Data Store.


Total precipitation is requested on the monthly time step, while 2m temperature is requested on the hourly timestep, because the monthly data in GEE is created from the hourly timesteps. The data on years 2020-2023 is requested because these years are not present in GEE ERA5 dataset.

**NB!**

A .cdsapirc file is needed to access your user (like the json access key on the bucket). To create it, go to https://cds.climate.copernicus.eu/profile (while logged in with your used credentials), scroll down to API key and copy the "url: xxxx key: xxx" section. Save it locally on your computer with the name ".cdsapirc" under "C:\Users\<your-username>" or provide a custom path with os.environ["CDSAPI_RC"] = r"C:\path\to\.cdsapirc"

In [ ]:
# specify file paths where data is downloaded to. 

os.chdir("..\\") 

cwd = os.getcwd()

output_folder = f"{cwd}\\era5_input_rasters\\"

# Creates folders or skips if they exist
Path(output_folder).mkdir(parents=True, exist_ok=True)

In [ ]:
region = 'est'

In [ ]:
# Specify area of interest for ERA5 data download
# area = [north, west, south, east]

In [ ]:
# if region == 'bal':
    
#     area = [70, 10, 45, 40]
    
# elif region == 'eur':
    
#     area = [81, -30, 26, 57]

# else:
    
#     sys.exit(f"No area specified for region {region} yet.")

In [ ]:
# Specify the requested parameters, dates, area (comment out unwanted parameters)

monthly_parameters = ['evaporation', "total_precipitation"]

hourly_parameters =  ['2m_temperature'] # will calculate mean, min, max from this


years = [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

months = list(range(1, 13, 1)) # all months of the year

In [ ]:
### functions

In [ ]:
def download_era5_monthly_data(year, month, parameter, output_file):
    """Dowloads the ERA5 reanalysis data on a monthly timestep for a 
    specified parameter, year, month, area.
    Saves the file as an unarchived .nc file"""
    
    dataset = "reanalysis-era5-single-levels-monthly-means"
    
    # Set up the CDS API client
    c = cdsapi.Client()
    
    request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": [parameter],
    "year": [f"{year}"],
    "month": [f"{month:02d}"],
    "time": ["00:00"],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": area}
    
    # Submit the request to download the data
    print(f"Requesting data for {year}-{month:02d}...")
    
    c.retrieve(dataset, request, output_file)
    print(f"Download complete: {output_file}")

In [ ]:
def download_era5_hourly_data(year, month, hourly_parameter, output_file):
    """Downloads ERA5 reanalysis data on hourly timestep for 
    a specified parameter, year, month, area.
    Saves it as an unarchived .nc file"""
    
    dataset = "reanalysis-era5-single-levels"
    
    # Set up the CDS API client
    c = cdsapi.Client()
    
    # Get number of days in the month on the fly
    days_in_month = calendar.monthrange(year, month)[1] 
    days = [f"{day:02d}" for day in range(1, days_in_month + 1)]
    
    # format request
    request = {
    "product_type": ["reanalysis"],
    "variable": hourly_parameter,
    "year": [f"{year}"],
    "month": [f"{month:02d}"],
    "day": days, 
    "time": [
        "00:00", "01:00", "02:00",
        "03:00", "04:00", "05:00",
        "06:00", "07:00", "08:00",
        "09:00", "10:00", "11:00",
        "12:00", "13:00", "14:00",
        "15:00", "16:00", "17:00",
        "18:00", "19:00", "20:00",
        "21:00", "22:00", "23:00"
    ],
    "data_format": "netcdf",
    "download_format": "unarchived",
    "area": area
}
    # Submit the request to download the data
    print(f"Requesting data for {year}-{month:02d}...")
    
    c.retrieve(dataset, request, output_file)
    print(f"Download complete: {output_file}")
    

In [ ]:
### function calls

In [ ]:
start_time = time.time()

In [ ]:
for parameter in monthly_parameters:
    
    for year in years:
        
        for month in months:
            
            output_file = f"{output_folder}era5_{parameter}_monthly_{year}_{month}.nc"
            
            
            #download_era5_monthly_data(year, month, parameter, output_file)
            

In [ ]:
for parameter in hourly_parameters:
    
    for year in years:
        
        for month in months:
            
            output_file = f"{output_folder}//era5_temperature_hourly_{year}_{month:02d}.nc"
            
            
            download_era5_hourly_data(year, month, parameter, output_file)
            

In [ ]:
end_time = time.time()

In [ ]:
elapsed_time = end_time - start_time

elapsed_hours = elapsed_time / 3600

print(f"Elapsed time: {elapsed_time:.4f} seconds ( {elapsed_hours:.2f} hours )")

In [ ]:
print(f" Exported on {datetime.today().strftime('%Y-%m-%d')}")